# Model Serving metrics setup

This notebook configures a continuous serverless collector and publishes an AI/BI dashboard for Model Serving endpoint metrics. It discovers all non-foundation workspace endpoints when the include list is empty and adds inference-table datasets when payload logging is already available.

Place overrides in a `.env` file beside this notebook, then run the notebook from top to bottom.

In [ ]:
# A sibling .env file overrides these internal code defaults.
# Inference tables are detected but never enabled by this setup.
SETUP_DEFAULTS = {
    "INCLUDE_ENDPOINT_NAMES": "",
    "STORAGE_CATALOG": spark.sql("SELECT current_catalog()").first()[0],
    "STORAGE_SCHEMA": "model_serving_observability",
    "JOB_NAME": "Model serving metrics sync",
    "JOB_ID": "",
    "DASHBOARD_NAME": "Model Serving metrics",
    "DASHBOARD_ID": "",
    "COLLECTOR_NOTEBOOK_NAME": "model_serving_metrics_collector",
    "COLLECTION_INTERVAL_SECONDS": "60",
    "ENDPOINT_REFRESH_CYCLES": "60",
    "COLLECTION_PAUSE_STATUS": "UNPAUSED",
    "WAREHOUSE_ID": "",
    "PARENT_PATH": "",
    "COLLECTOR_PERFORMANCE_TARGET": "STANDARD",
    "COLLECTOR_ENVIRONMENT_KEY": "collector",
    "COLLECTOR_ENVIRONMENT_VERSION": "4",
    "COLLECTOR_DEPENDENCIES": '"prometheus-client>=0.20,<1", databricks-sdk>=0.36.0',
    "DASHBOARD_LOOKBACK_HOURS": "24",
}

## Discover workspace resources

Resolve endpoint names, validate continuous collection settings, detect inference tables, select a SQL warehouse, and derive workspace paths.

In [ ]:
import base64
import csv
import hashlib
import json
import re
import shlex
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import quote

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import BadRequest, NotFound


LOG_NAME = "serving_metrics_setup"


def _print_log(event: str, level: str = "INFO", **fields: Any) -> None:
    """Write a log-library-style line with JSON fields or a plain text event."""
    timestamp = datetime.now(timezone.utc).isoformat()
    payload = (
        json.dumps(
            {"event": event, **fields},
            default=str,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        if fields
        else event
    )
    print(f"[{level.upper()}] {LOG_NAME} {timestamp} - {payload}", flush=True)


def _read_env_file(notebook_path: str) -> tuple[Path | None, dict[str, str]]:
    """Read the first sibling or working-directory .env file that exists."""
    workspace_path = Path(
        notebook_path
        if notebook_path.startswith("/Workspace/")
        else f"/Workspace{notebook_path}"
    )
    candidates = [workspace_path.parent / ".env", Path.cwd() / ".env"]
    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if not candidate.is_file():
            continue
        values: dict[str, str] = {}
        for line_number, raw_line in enumerate(candidate.read_text().splitlines(), start=1):
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[7:].strip()
            if "=" not in line:
                raise ValueError(
                    f"Invalid .env line {line_number} in {candidate}: {raw_line!r}"
                )
            name, value = line.split("=", 1)
            name = name.strip()
            try:
                value_parts = shlex.split(value, comments=True, posix=True)
            except ValueError as error:
                raise ValueError(
                    f"Invalid .env value on line {line_number} in {candidate}: {raw_line!r}"
                ) from error
            values[name] = " ".join(value_parts)
        return candidate, values
    return None, {}


def _resolve_config(
    defaults: dict[str, str],
    env_values: dict[str, str],
) -> tuple[dict[str, str], dict[str, str]]:
    """Resolve configuration from .env values and internal defaults."""
    values = {
        name: env_values.get(name, code_default).strip()
        for name, code_default in defaults.items()
    }
    sources = {
        name: ".env" if name in env_values else "code_default"
        for name in defaults
    }
    return values, sources


def _csv_string_list(name: str, value: str) -> list[str]:
    """Parse CSV list values, trimming whitespace and omitting empty entries."""
    text = value.strip()
    if not text or text == "[]":
        return []
    if text.startswith("["):
        try:
            legacy_values = json.loads(text)
        except json.JSONDecodeError as error:
            raise ValueError(f"{name} must be CSV") from error
        if not isinstance(legacy_values, list) or not all(
            isinstance(item, str) for item in legacy_values
        ):
            raise ValueError(f"{name} must be CSV")
        return [item.strip() for item in legacy_values if item.strip()]
    try:
        values = next(csv.reader([text], skipinitialspace=True))
    except csv.Error as error:
        raise ValueError(f"{name} must be valid CSV") from error
    return [item.strip() for item in values if item.strip()]


SETUP_NOTEBOOK_PATH = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
ENV_FILE_PATH, ENV_FILE_VALUES = _read_env_file(SETUP_NOTEBOOK_PATH)
recognized_env_values = {
    name: value
    for name, value in ENV_FILE_VALUES.items()
    if name in SETUP_DEFAULTS
}
unknown_env_names = sorted(set(ENV_FILE_VALUES) - set(SETUP_DEFAULTS))
SETUP_VALUES, SETUP_VALUE_SOURCES = _resolve_config(
    SETUP_DEFAULTS,
    recognized_env_values,
)
INCLUDE_ENDPOINT_NAMES = _csv_string_list(
    "INCLUDE_ENDPOINT_NAMES",
    SETUP_VALUES["INCLUDE_ENDPOINT_NAMES"],
)
STORAGE_CATALOG = SETUP_VALUES["STORAGE_CATALOG"]
STORAGE_SCHEMA = SETUP_VALUES["STORAGE_SCHEMA"]
JOB_NAME = SETUP_VALUES["JOB_NAME"]
JOB_ID = SETUP_VALUES["JOB_ID"]
DASHBOARD_NAME = SETUP_VALUES["DASHBOARD_NAME"]
DASHBOARD_ID = SETUP_VALUES["DASHBOARD_ID"]
COLLECTOR_NOTEBOOK_NAME = SETUP_VALUES["COLLECTOR_NOTEBOOK_NAME"]
COLLECTION_INTERVAL_SECONDS = int(SETUP_VALUES["COLLECTION_INTERVAL_SECONDS"])
ENDPOINT_REFRESH_CYCLES = int(SETUP_VALUES["ENDPOINT_REFRESH_CYCLES"])
COLLECTION_PAUSE_STATUS = SETUP_VALUES["COLLECTION_PAUSE_STATUS"].upper()
WAREHOUSE_ID = SETUP_VALUES["WAREHOUSE_ID"]
PARENT_PATH = SETUP_VALUES["PARENT_PATH"]
COLLECTOR_PERFORMANCE_TARGET = SETUP_VALUES["COLLECTOR_PERFORMANCE_TARGET"].upper()
COLLECTOR_ENVIRONMENT_KEY = SETUP_VALUES["COLLECTOR_ENVIRONMENT_KEY"]
COLLECTOR_ENVIRONMENT_VERSION = SETUP_VALUES["COLLECTOR_ENVIRONMENT_VERSION"]
COLLECTOR_DEPENDENCIES = _csv_string_list(
    "COLLECTOR_DEPENDENCIES",
    SETUP_VALUES["COLLECTOR_DEPENDENCIES"],
)
DASHBOARD_LOOKBACK_HOURS = int(SETUP_VALUES["DASHBOARD_LOOKBACK_HOURS"])
workspace = WorkspaceClient()
WORKSPACE_HOST = workspace.config.host.rstrip("/")


def _is_foundation_endpoint(endpoint: Any) -> bool:
    """Return whether an endpoint routes to a system.ai foundation model."""
    endpoint_dict = endpoint.as_dict()
    config = endpoint_dict.get("config") or {}
    entities = config.get("served_entities") or config.get("served_models") or []
    return any(
        str(entity.get("entity_name") or entity.get("model_name") or "").startswith("system.ai.")
        for entity in entities
    )


def _resolve_endpoint_names(configured_names: str | list[str] | None) -> list[str]:
    """Validate an optional include list or discover all non-foundation endpoints."""
    if isinstance(configured_names, list):
        names = [name.strip() for name in configured_names if name.strip()]
    else:
        text = (configured_names or "").strip()
        if text.startswith("["):
            parsed = json.loads(text)
            if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):
                raise ValueError("JSON INCLUDE_ENDPOINT_NAMES must be an array of strings")
            names = [name.strip() for name in parsed if name.strip()]
        else:
            names = [name.strip() for name in text.split(",") if name.strip()]

    endpoints = [endpoint for endpoint in workspace.serving_endpoints.list() if endpoint.name]
    endpoint_by_name = {endpoint.name: endpoint for endpoint in endpoints}
    foundation_names = {
        endpoint.name for endpoint in endpoints if _is_foundation_endpoint(endpoint)
    }
    if names:
        requested_names = set(names)
        missing_names = sorted(requested_names - endpoint_by_name.keys())
        if missing_names:
            raise ValueError(f"Included serving endpoints were not found: {missing_names}")
        blocked_names = sorted(requested_names & foundation_names)
        if blocked_names:
            raise ValueError(
                "Foundation Model API endpoints cannot be collected by export-metrics: "
                f"{blocked_names}"
            )
        resolved_names = sorted(requested_names)
        _print_log("endpoint_include_resolved", endpoints=resolved_names)
        return resolved_names

    resolved_names = sorted(endpoint_by_name.keys() - foundation_names)
    _print_log(
        "endpoint_discovery_complete",
        endpoint_count=len(resolved_names),
        endpoints=resolved_names,
        skipped_foundation_count=len(foundation_names),
        skipped_foundation_endpoints=sorted(foundation_names),
    )
    return resolved_names


def _configured_inference_table(endpoint_name: str) -> str | None:
    """Read one endpoint's API configuration and derive its payload table name."""
    endpoint = workspace.serving_endpoints.get(endpoint_name).as_dict()
    gateway_config = (endpoint.get("ai_gateway") or {}).get("inference_table_config") or {}
    if gateway_config.get("enabled"):
        catalog_name = gateway_config.get("catalog_name")
        schema_name = gateway_config.get("schema_name")
        table_prefix = gateway_config.get("table_name_prefix")
        if catalog_name and schema_name and table_prefix:
            return f"{catalog_name}.{schema_name}.{table_prefix}_payload"

    legacy_config = (endpoint.get("config") or {}).get("auto_capture_config") or {}
    if not legacy_config.get("enabled"):
        return None

    payload_table = ((legacy_config.get("state") or {}).get("payload_table") or {}).get("name")
    if payload_table:
        return payload_table

    catalog_name = legacy_config.get("catalog_name")
    schema_name = legacy_config.get("schema_name")
    table_prefix = legacy_config.get("table_name_prefix")
    if catalog_name and schema_name and table_prefix:
        return f"{catalog_name}.{schema_name}.{table_prefix}_payload"
    return None


def _validate_metrics_export(endpoint_name: str) -> None:
    """Fail when an explicitly included endpoint cannot export metrics."""
    response = workspace.api_client.do(
        "GET",
        f"/api/2.0/serving-endpoints/{quote(endpoint_name, safe='')}/metrics",
        headers={"Accept": "text/plain"},
        raw=True,
    )
    contents = response.get("contents") if isinstance(response, dict) else None
    payload = contents.read() if hasattr(contents, "read") else contents
    if not payload:
        raise RuntimeError(f"Metrics export returned no content for endpoint {endpoint_name!r}")
    _print_log(
        "endpoint_metrics_export_validated",
        endpoint_name=endpoint_name,
        response_bytes=len(payload),
    )


def _sql_identifier(identifier: str) -> str:
    """Quote a three-part Unity Catalog identifier."""
    return ".".join(f"`{part.replace('`', '``')}`" for part in identifier.split("."))


def _sql_literal(value: str) -> str:
    """Quote a SQL string literal."""
    return "'" + value.replace("'", "''") + "'"


for setting_name, setting_value in {
    "JOB_NAME": JOB_NAME,
    "DASHBOARD_NAME": DASHBOARD_NAME,
    "COLLECTOR_NOTEBOOK_NAME": COLLECTOR_NOTEBOOK_NAME,
    "COLLECTOR_ENVIRONMENT_KEY": COLLECTOR_ENVIRONMENT_KEY,
    "COLLECTOR_ENVIRONMENT_VERSION": COLLECTOR_ENVIRONMENT_VERSION,
}.items():
    if not setting_value.strip():
        raise ValueError(f"{setting_name} cannot be empty")
if not COLLECTOR_DEPENDENCIES:
    raise ValueError("COLLECTOR_DEPENDENCIES must contain collector runtime dependencies")

identifier_pattern = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
for identifier_name, identifier_value in {
    "STORAGE_CATALOG": STORAGE_CATALOG,
    "STORAGE_SCHEMA": STORAGE_SCHEMA,
}.items():
    if not identifier_pattern.fullmatch(identifier_value):
        raise ValueError(
            f"{identifier_name} must contain only letters, numbers, and underscores: "
            f"{identifier_value!r}"
        )

if COLLECTION_PAUSE_STATUS not in {"PAUSED", "UNPAUSED"}:
    raise ValueError("COLLECTION_PAUSE_STATUS must be PAUSED or UNPAUSED")
if COLLECTOR_PERFORMANCE_TARGET not in {"STANDARD", "PERFORMANCE_OPTIMIZED"}:
    raise ValueError("COLLECTOR_PERFORMANCE_TARGET must be STANDARD or PERFORMANCE_OPTIMIZED")
if COLLECTION_INTERVAL_SECONDS < 10:
    raise ValueError("COLLECTION_INTERVAL_SECONDS must be at least 10")
if ENDPOINT_REFRESH_CYCLES < 1:
    raise ValueError("ENDPOINT_REFRESH_CYCLES must be at least 1")
if DASHBOARD_LOOKBACK_HOURS < 1:
    raise ValueError("DASHBOARD_LOOKBACK_HOURS must be at least 1")
available_catalogs = {row[0] for row in spark.sql("SHOW CATALOGS").collect()}
if STORAGE_CATALOG not in available_catalogs:
    raise RuntimeError(
        f"Unity Catalog catalog was not found: {STORAGE_CATALOG}. "
        f"Available catalogs: {sorted(available_catalogs)}"
    )
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {STORAGE_CATALOG}.{STORAGE_SCHEMA}")
_print_log(
    "unity_catalog_destination_validated",
    catalog=STORAGE_CATALOG,
    schema=STORAGE_SCHEMA,
)

RESOLVED_ENDPOINT_NAMES = _resolve_endpoint_names(INCLUDE_ENDPOINT_NAMES)
if not RESOLVED_ENDPOINT_NAMES:
    raise RuntimeError("The workspace contains no collectable Model Serving endpoints")
if INCLUDE_ENDPOINT_NAMES:
    for endpoint_name in RESOLVED_ENDPOINT_NAMES:
        _validate_metrics_export(endpoint_name)

CONFIGURED_INFERENCE_TABLES = {
    endpoint_name: table_name
    for endpoint_name in RESOLVED_ENDPOINT_NAMES
    if (table_name := _configured_inference_table(endpoint_name)) is not None
}
AVAILABLE_INFERENCE_TABLES = {
    endpoint_name: table_name
    for endpoint_name, table_name in CONFIGURED_INFERENCE_TABLES.items()
    if spark.catalog.tableExists(table_name)
}
if AVAILABLE_INFERENCE_TABLES:
    INFERENCE_TABLE_STATUS = "available"
elif CONFIGURED_INFERENCE_TABLES:
    INFERENCE_TABLE_STATUS = "configured"
else:
    INFERENCE_TABLE_STATUS = "unavailable"
missing_inference_tables = {
    endpoint_name: table_name
    for endpoint_name, table_name in CONFIGURED_INFERENCE_TABLES.items()
    if endpoint_name not in AVAILABLE_INFERENCE_TABLES
}
if missing_inference_tables:
    _print_log(
        "inference_tables_missing",
        level="warning",
        tables=missing_inference_tables,
    )
_print_log(
    "inference_table_discovery_complete",
    status=INFERENCE_TABLE_STATUS,
    configured_tables=CONFIGURED_INFERENCE_TABLES,
    available_tables=AVAILABLE_INFERENCE_TABLES,
)

notebook_path = SETUP_NOTEBOOK_PATH
resolved_parent_path = PARENT_PATH.strip().rstrip("/") or str(Path(notebook_path).parent)
collector_notebook_path = str(Path(notebook_path).with_name(COLLECTOR_NOTEBOOK_NAME))
if not collector_notebook_path.startswith("/"):
    collector_notebook_path = f"/{collector_notebook_path.lstrip('/')}"
collector_status = workspace.api_client.do(
    "GET",
    "/api/2.0/workspace/get-status",
    query={"path": collector_notebook_path},
)
if collector_status.get("object_type") != "NOTEBOOK":
    raise RuntimeError(
        f"Collector notebook was not found at {collector_notebook_path}: {collector_status}"
    )
collector_export = workspace.api_client.do(
    "GET",
    "/api/2.0/workspace/export",
    query={"path": collector_notebook_path, "format": "JUPYTER"},
)
collector_content = collector_export.get("content")
if not collector_content:
    raise RuntimeError(f"Collector notebook export returned no content: {collector_notebook_path}")
COLLECTOR_CODE_HASH = hashlib.sha256(
    base64.b64decode(collector_content)
).hexdigest()[:16]
parent_status = workspace.api_client.do(
    "GET",
    "/api/2.0/workspace/get-status",
    query={"path": resolved_parent_path},
)
if parent_status.get("object_type") != "DIRECTORY":
    raise RuntimeError(
        f"Dashboard parent path is not a workspace directory: {resolved_parent_path}"
    )
_print_log(
    "workspace_paths_validated",
    setup_notebook_path=notebook_path,
    collector_notebook_path=collector_notebook_path,
    collector_code_hash=COLLECTOR_CODE_HASH,
    dashboard_parent_path=resolved_parent_path,
)

if WAREHOUSE_ID:
    resolved_warehouse_id = WAREHOUSE_ID
else:
    warehouses = [warehouse for warehouse in workspace.warehouses.list() if warehouse.id]
    if not warehouses:
        raise RuntimeError("No SQL warehouse is available. Set WAREHOUSE_ID in the first cell.")
    warehouses.sort(
        key=lambda warehouse: (
            str(getattr(warehouse, "state", "")) != "State.RUNNING",
            not bool(getattr(warehouse, "enable_serverless_compute", False)),
            str(getattr(warehouse, "name", "")),
        )
    )
    resolved_warehouse_id = warehouses[0].id
resolved_warehouse = workspace.warehouses.get(resolved_warehouse_id)
_print_log(
    "sql_warehouse_validated",
    warehouse_id=resolved_warehouse_id,
    warehouse_name=resolved_warehouse.name,
    warehouse_state=str(resolved_warehouse.state),
    serverless=resolved_warehouse.enable_serverless_compute,
)

TABLE_PREFIX = f"{STORAGE_CATALOG}.{STORAGE_SCHEMA}"
ENDPOINT_SNAPSHOTS_TABLE = f"{TABLE_PREFIX}.model_serving_endpoint_snapshots"
RAW_METRICS_TABLE = f"{TABLE_PREFIX}.model_serving_metrics_raw"
METRIC_SAMPLES_TABLE = f"{TABLE_PREFIX}.model_serving_metric_samples"
COUNTER_RATES_VIEW = f"{TABLE_PREFIX}.model_serving_counter_rates"
HISTOGRAM_PERCENTILES_VIEW = f"{TABLE_PREFIX}.model_serving_histogram_percentiles"
DASHBOARD_SERIES_VIEW = f"{TABLE_PREFIX}.model_serving_dashboard_series"
_print_log(
    "setup_configuration_resolved",
    env_file_path=str(ENV_FILE_PATH) if ENV_FILE_PATH else None,
    ignored_env_names=unknown_env_names,
    value_sources=SETUP_VALUE_SOURCES,
    include_endpoint_names=INCLUDE_ENDPOINT_NAMES,
    resolved_endpoint_names=RESOLVED_ENDPOINT_NAMES,
    storage_catalog=STORAGE_CATALOG,
    storage_schema=STORAGE_SCHEMA,
    endpoint_snapshots_table=ENDPOINT_SNAPSHOTS_TABLE,
    raw_metrics_table=RAW_METRICS_TABLE,
    metric_samples_table=METRIC_SAMPLES_TABLE,
    counter_rates_view=COUNTER_RATES_VIEW,
    histogram_percentiles_view=HISTOGRAM_PERCENTILES_VIEW,
    dashboard_series_view=DASHBOARD_SERIES_VIEW,
    job_name=JOB_NAME,
    job_id=JOB_ID or None,
    dashboard_name=DASHBOARD_NAME,
    dashboard_id=DASHBOARD_ID or None,
    setup_notebook_path=notebook_path,
    collector_notebook_path=collector_notebook_path,
    collection_interval_seconds=COLLECTION_INTERVAL_SECONDS,
    endpoint_refresh_cycles=ENDPOINT_REFRESH_CYCLES,
    collection_pause_status=COLLECTION_PAUSE_STATUS,
    sql_warehouse_id=resolved_warehouse_id,
    sql_warehouse_name=resolved_warehouse.name,
    dashboard_parent_path=resolved_parent_path,
    collector_performance_target=COLLECTOR_PERFORMANCE_TARGET,
    collector_environment_key=COLLECTOR_ENVIRONMENT_KEY,
    collector_environment_version=COLLECTOR_ENVIRONMENT_VERSION,
    collector_dependencies=COLLECTOR_DEPENDENCIES,
    dashboard_lookback_hours=DASHBOARD_LOOKBACK_HOURS,
    inference_table_status=INFERENCE_TABLE_STATUS,
    configured_inference_tables=CONFIGURED_INFERENCE_TABLES,
    available_inference_tables=AVAILABLE_INFERENCE_TABLES,
    workspace_host=WORKSPACE_HOST,
)

## Configure the continuous collector

Create or update the continuous serverless Lakeflow Job and wait for its Delta views when they do not exist yet.

In [ ]:
collector_configuration = {
    "collector_code_hash": COLLECTOR_CODE_HASH,
    "include_endpoint_names": INCLUDE_ENDPOINT_NAMES,
    "storage_catalog": STORAGE_CATALOG,
    "storage_schema": STORAGE_SCHEMA,
    "poll_interval_seconds": COLLECTION_INTERVAL_SECONDS,
    "endpoint_refresh_cycles": ENDPOINT_REFRESH_CYCLES,
}
collector_configuration_hash = hashlib.sha256(
    json.dumps(collector_configuration, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
job_settings = {
    "name": JOB_NAME,
    "description": (
        "Continuously scrapes Model Serving Prometheus metrics into Delta and discovers "
        "configured inference tables from each endpoint API."
    ),
    "tags": {
        "collector_code_hash": COLLECTOR_CODE_HASH,
        "configuration_hash": collector_configuration_hash,
        "inference_tables": INFERENCE_TABLE_STATUS,
    },
    "performance_target": COLLECTOR_PERFORMANCE_TARGET,
    "max_concurrent_runs": 1,
    "continuous": {
        "pause_status": COLLECTION_PAUSE_STATUS,
        "task_retry_mode": "ON_FAILURE",
    },
    "parameters": [],
    "environments": [
        {
            "environment_key": COLLECTOR_ENVIRONMENT_KEY,
            "spec": {
                "environment_version": COLLECTOR_ENVIRONMENT_VERSION,
                "dependencies": COLLECTOR_DEPENDENCIES,
            },
        }
    ],
    "tasks": [
        {
            "task_key": "scrape_serving_metrics",
            "environment_key": COLLECTOR_ENVIRONMENT_KEY,
            "notebook_task": {
                "notebook_path": collector_notebook_path,
                "source": "WORKSPACE",
                "base_parameters": {},
            },
        }
    ],
}

def _configuration_differences(
    actual: Any,
    expected: Any,
    path: str = "settings",
) -> list[dict[str, Any]]:
    """Compare only fields owned by this setup and ignore server-added fields."""
    if isinstance(expected, dict):
        if not expected:
            return [] if actual in (None, {}) else [
                {"path": path, "expected": expected, "actual": actual}
            ]
        if not isinstance(actual, dict):
            return [{"path": path, "expected": expected, "actual": actual}]
        differences: list[dict[str, Any]] = []
        for key, expected_value in expected.items():
            differences.extend(
                _configuration_differences(
                    actual.get(key),
                    expected_value,
                    f"{path}.{key}",
                )
            )
        return differences
    if isinstance(expected, list):
        if not expected:
            return [] if actual in (None, []) else [
                {"path": path, "expected": expected, "actual": actual}
            ]
        if not isinstance(actual, list) or len(actual) != len(expected):
            return [{"path": path, "expected": expected, "actual": actual}]
        differences = []
        for index, expected_value in enumerate(expected):
            differences.extend(
                _configuration_differences(
                    actual[index],
                    expected_value,
                    f"{path}[{index}]",
                )
            )
        return differences
    if actual != expected:
        return [{"path": path, "expected": expected, "actual": actual}]
    return []


def _get_job_settings(resolved_job_id: int) -> dict[str, Any]:
    response = workspace.api_client.do(
        "GET",
        "/api/2.1/jobs/get",
        query={"job_id": resolved_job_id},
    )
    return response.get("settings") or {}


def _wait_for_job_settings(resolved_job_id: int) -> list[dict[str, Any]]:
    differences: list[dict[str, Any]] = []
    for attempt in range(1, 7):
        differences = _configuration_differences(
            _get_job_settings(resolved_job_id),
            job_settings,
        )
        _print_log(
            "job_settings_verification",
            attempt=attempt,
            job_id=resolved_job_id,
            differences=differences,
        )
        if not differences:
            return []
        time.sleep(2)
    return differences


def _create_job() -> int:
    response = workspace.api_client.do("POST", "/api/2.1/jobs/create", body=job_settings)
    created_job_id = response.get("job_id")
    if not created_job_id:
        raise RuntimeError(f"Job create response did not contain job_id: {response}")
    return int(created_job_id)


if JOB_ID:
    try:
        existing_job_id = int(JOB_ID)
    except ValueError as error:
        raise ValueError("JOB_ID must be an integer string") from error
    _get_job_settings(existing_job_id)
else:
    matching_jobs = list(workspace.jobs.list(name=JOB_NAME))
    if len(matching_jobs) > 1:
        raise RuntimeError(
            f"Multiple jobs named {JOB_NAME!r} exist. Set JOB_ID in the first cell. "
            f"Matches: {[job.job_id for job in matching_jobs]}"
        )
    existing_job_id = int(matching_jobs[0].job_id) if matching_jobs else None

job_differences = (
    _configuration_differences(_get_job_settings(existing_job_id), job_settings)
    if existing_job_id is not None
    else []
)
_print_log(
    "job_configuration_resolved",
    job_name=JOB_NAME,
    existing_job_id=existing_job_id,
    differences=job_differences,
    collector_configuration=collector_configuration,
    collector_configuration_hash=collector_configuration_hash,
    continuous=job_settings["continuous"],
    environment=job_settings["environments"][0],
    performance_target=COLLECTOR_PERFORMANCE_TARGET,
)
job_change_started_at_ms = (
    int(time.time() * 1000)
    if existing_job_id is None or job_differences
    else None
)
if existing_job_id is None:
    job_id = _create_job()
    job_action = "created"
elif not job_differences:
    job_id = existing_job_id
    job_action = "unchanged"
else:
    try:
        workspace.api_client.do(
            "POST",
            "/api/2.1/jobs/reset",
            body={"job_id": existing_job_id, "new_settings": job_settings},
        )
        remaining_differences = _wait_for_job_settings(existing_job_id)
    except BadRequest as error:
        remaining_differences = [{"path": "reset", "error": str(error)}]
    if remaining_differences:
        _print_log(
            "job_rebuild_required",
            level="warning",
            old_job_id=existing_job_id,
            differences=remaining_differences,
        )
        workspace.api_client.do(
            "POST",
            "/api/2.1/jobs/delete",
            body={"job_id": existing_job_id},
        )
        job_id = _create_job()
        job_action = "rebuilt"
    else:
        job_id = existing_job_id
        job_action = "updated"

final_job_differences = _wait_for_job_settings(job_id)
if final_job_differences:
    raise RuntimeError(
        "Job settings did not converge after reconciliation: "
        + json.dumps(final_job_differences, sort_keys=True)
    )
_print_log("job_configuration_applied", action=job_action, job_id=job_id)

def _continuous_runs(active_only: bool) -> list[dict[str, Any]]:
    response = workspace.api_client.do(
        "GET",
        "/api/2.1/jobs/runs/list",
        query={"job_id": job_id, "limit": 10, "active_only": active_only},
    )
    return [
        run
        for run in response.get("runs", [])
        if run.get("trigger") == "CONTINUOUS"
    ]


def _get_run(run_id: int) -> dict[str, Any]:
    return workspace.api_client.do(
        "GET",
        "/api/2.1/jobs/runs/get",
        query={"run_id": run_id},
    )


def _latest_continuous_run() -> dict[str, Any] | None:
    runs = _continuous_runs(active_only=False)
    if not runs or not runs[0].get("run_id"):
        return None
    return _get_run(int(runs[0]["run_id"]))


def _run_has_parameter_overrides(run: dict[str, Any]) -> bool:
    if run.get("job_parameters"):
        return True
    return any(
        bool((task.get("notebook_task") or {}).get("base_parameters"))
        for task in run.get("tasks", [])
    )


def _cancel_stale_continuous_runs() -> list[int]:
    cancelled_run_ids: list[int] = []
    for listed_run in _continuous_runs(active_only=True):
        run_id = listed_run.get("run_id")
        if not run_id:
            continue
        run = _get_run(int(run_id))
        started_before_change = (
            job_change_started_at_ms is not None
            and int(run.get("start_time") or 0) < job_change_started_at_ms - 5000
        )
        if not started_before_change and not _run_has_parameter_overrides(run):
            continue
        _print_log(
            "stale_continuous_run_cancelling",
            job_id=job_id,
            run_id=run_id,
            run_start_time=run.get("start_time"),
            job_change_started_at_ms=job_change_started_at_ms,
            has_parameter_overrides=_run_has_parameter_overrides(run),
        )
        workspace.api_client.do(
            "POST",
            "/api/2.2/jobs/runs/cancel",
            body={"run_id": run_id},
        )
        cancelled_run_ids.append(int(run_id))
    return cancelled_run_ids


cancelled_stale_run_ids = _cancel_stale_continuous_runs()
_print_log(
    "stale_continuous_run_reconciliation_complete",
    job_id=job_id,
    cancelled_run_ids=cancelled_stale_run_ids,
)


def _dashboard_data_ready() -> bool:
    if not spark.catalog.tableExists(DASHBOARD_SERIES_VIEW):
        return False
    endpoint_names_sql = ", ".join(
        _sql_literal(endpoint_name) for endpoint_name in RESOLVED_ENDPOINT_NAMES
    )
    rows = spark.sql(
        f"""
        SELECT 1
        FROM {DASHBOARD_SERIES_VIEW}
        WHERE endpoint_name IN ({endpoint_names_sql})
          AND chart_name IS NOT NULL
        LIMIT 1
        """
    ).collect()
    return bool(rows)


if COLLECTION_PAUSE_STATUS == "PAUSED":
    if not _dashboard_data_ready():
        raise RuntimeError(
            "The collector is PAUSED and dashboard data is unavailable for the resolved endpoints. "
            "Set COLLECTION_PAUSE_STATUS to UNPAUSED for initial setup."
        )
    _print_log("collector_paused", job_id=job_id, dashboard_data_ready=True)
else:
    for attempt in range(1, 31):
        continuous_run = _latest_continuous_run()
        run_state = (continuous_run or {}).get("state") or {}
        life_cycle_state = run_state.get("life_cycle_state", "PENDING")
        result_state = run_state.get("result_state")
        state_message = run_state.get("state_message", "")
        run_start_time = int((continuous_run or {}).get("start_time") or 0)
        run_configuration_current = (
            (
                job_change_started_at_ms is None
                or run_start_time >= job_change_started_at_ms - 5000
            )
            and not _run_has_parameter_overrides(continuous_run or {})
        )
        task_statuses = [
            {
                "task_key": task.get("task_key"),
                "life_cycle_state": (task.get("state") or {}).get("life_cycle_state", "PENDING"),
                "result_state": (task.get("state") or {}).get("result_state"),
                "state_message": (task.get("state") or {}).get("state_message", ""),
            }
            for task in (continuous_run or {}).get("tasks", [])
        ]
        task_running = any(
            task_status["life_cycle_state"] == "RUNNING"
            for task_status in task_statuses
        )
        task_failures = [
            task_status
            for task_status in task_statuses
            if task_status["result_state"] in {"FAILED", "TIMED_OUT"}
        ]
        dashboard_data_ready = _dashboard_data_ready()
        _print_log(
            "collector_startup_poll",
            attempt=attempt,
            job_id=job_id,
            run_id=(continuous_run or {}).get("run_id"),
            life_cycle_state=life_cycle_state,
            result_state=result_state,
            state_message=state_message,
            configuration_hash=collector_configuration_hash,
            job_change_started_at_ms=job_change_started_at_ms,
            run_start_time=run_start_time,
            run_configuration_current=run_configuration_current,
            task_statuses=task_statuses,
            task_running=task_running,
            task_failures=task_failures,
            dashboard_data_ready=dashboard_data_ready,
        )
        if run_configuration_current and task_failures and not task_running:
            if attempt < 3:
                time.sleep(10)
                continue
            raise RuntimeError(
                "Continuous collector task failed during startup: "
                + json.dumps(task_failures, sort_keys=True)
            )
        if run_configuration_current and (
            life_cycle_state in {"INTERNAL_ERROR", "SKIPPED"}
            or (life_cycle_state == "TERMINATED" and result_state != "SUCCESS")
        ):
            if attempt < 3:
                time.sleep(10)
                continue
            raise RuntimeError(
                f"Continuous collector failed during startup: state={life_cycle_state} "
                f"result={result_state} message={state_message}"
            )
        if (
            life_cycle_state == "RUNNING"
            and task_running
            and run_configuration_current
            and dashboard_data_ready
        ):
            break
        time.sleep(10)
    else:
        raise TimeoutError(
            f"Continuous collector job {job_id} did not start with the resolved configuration "
            "and produce dashboard data within 5 minutes"
        )

## Prepare dashboard datasets

Define reusable dashboard widgets, build the rolling metrics queries, and validate every dataset query before publication.

In [ ]:
def _query_lines(sql: str) -> list[str]:
    """Split SQL into Lakeview query lines while preserving separators."""
    return [f"{line}\n" for line in sql.strip().splitlines()]


def _text_widget(name: str, text: str, x: int, y: int, width: int, height: int) -> dict:
    return {
        "widget": {"name": name, "multilineTextboxSpec": {"lines": [text]}},
        "position": {"x": x, "y": y, "width": width, "height": height},
    }


def _counter_widget(
    dataset_name: str,
    time_column: str,
    name: str,
    title: str,
    field_name: str,
    expression: str,
    x: int,
    y: int,
    format_spec: dict[str, Any] | None = None,
) -> dict:
    value_encoding: dict[str, Any] = {"fieldName": field_name, "displayName": title}
    if format_spec:
        value_encoding["format"] = format_spec
    period_name = f"minute({time_column})"
    return {
        "widget": {
            "name": name,
            "queries": [
                {
                    "name": "main_query",
                    "query": {
                        "datasetName": dataset_name,
                        "fields": [
                            {"name": field_name, "expression": expression},
                            {
                                "name": period_name,
                                "expression": f'DATE_TRUNC("MINUTE", `{time_column}`)',
                            },
                        ],
                        "disaggregated": False,
                    },
                }
            ],
            "spec": {
                "version": 2,
                "widgetType": "counter",
                "encodings": {
                    "value": value_encoding,
                    "period": {"fieldName": period_name},
                },
                "frame": {"showTitle": True, "title": title},
            },
        },
        "position": {"x": x, "y": y, "width": 3, "height": 4},
    }


def _line_widget(
    dataset_name: str,
    time_column: str,
    name: str,
    title: str,
    y_fields: list[tuple[str, str, str]],
    x: int,
    y: int,
    format_spec: dict[str, Any] | None = None,
) -> dict:
    fields = [{"name": time_column, "expression": f"`{time_column}`"}]
    y_encodings: list[dict[str, Any]] = []
    for field_name, expression, display_name in y_fields:
        fields.append({"name": field_name, "expression": expression})
        encoding: dict[str, Any] = {"fieldName": field_name, "displayName": display_name}
        if format_spec:
            encoding["format"] = format_spec
        y_encodings.append(encoding)
    return {
        "widget": {
            "name": name,
            "queries": [
                {
                    "name": "main_query",
                    "query": {
                        "datasetName": dataset_name,
                        "fields": fields,
                        "disaggregated": False,
                    },
                }
            ],
            "spec": {
                "version": 3,
                "widgetType": "line",
                "encodings": {
                    "x": {
                        "fieldName": time_column,
                        "scale": {"type": "temporal"},
                        "displayName": "Timestamp",
                    },
                    "y": {"scale": {"type": "quantitative"}, "fields": y_encodings},
                },
                "frame": {"showTitle": True, "title": title},
            },
        },
        "position": {"x": x, "y": y, "width": 6, "height": 6},
    }


def _filter_widget(
    dataset_name: str,
    field_name: str,
    title: str,
    name: str,
    widget_type: str,
    x: int,
    y: int,
    width: int,
) -> dict:
    query_name = f"{dataset_name}_{field_name}"
    return {
        "widget": {
            "name": name,
            "queries": [
                {
                    "name": query_name,
                    "query": {
                        "datasetName": dataset_name,
                        "fields": [{"name": field_name, "expression": f"`{field_name}`"}],
                        "disaggregated": False,
                    },
                }
            ],
            "spec": {
                "version": 2,
                "widgetType": widget_type,
                "encodings": {
                    "fields": [
                        {
                            "fieldName": field_name,
                            "displayName": title,
                            "queryName": query_name,
                        }
                    ]
                },
                "frame": {"showTitle": True, "title": title},
            },
        },
        "position": {"x": x, "y": y, "width": width, "height": 2},
    }


def _inference_query() -> str | None:
    """Build request-level aggregates for every available inference table."""
    selects: list[str] = []
    for endpoint_name, table_name in AVAILABLE_INFERENCE_TABLES.items():
        columns = set(spark.table(table_name).columns)
        time_column = next(
            (column for column in ("event_time", "request_time", "timestamp") if column in columns),
            None,
        )
        latency_column = next(
            (
                column
                for column in ("latency_ms", "execution_duration_ms", "execution_time_ms")
                if column in columns
            ),
            None,
        )
        if not time_column or "status_code" not in columns:
            _print_log(
                "inference_table_schema_unsupported",
                level="warning",
                endpoint_name=endpoint_name,
                table_name=table_name,
                columns=sorted(columns),
            )
            continue

        latency_expression = (
            f"percentile_approx(CAST(`{latency_column}` AS DOUBLE), 0.50)"
            if latency_column
            else "CAST(NULL AS DOUBLE)"
        )
        latency_p99_expression = (
            f"percentile_approx(CAST(`{latency_column}` AS DOUBLE), 0.99)"
            if latency_column
            else "CAST(NULL AS DOUBLE)"
        )
        ttfb_expression = (
            "percentile_approx(CAST(`time_to_first_byte_ms` AS DOUBLE), 0.99)"
            if "time_to_first_byte_ms" in columns
            else "CAST(NULL AS DOUBLE)"
        )
        selects.append(
            f"""
SELECT
  DATE_TRUNC('MINUTE', `{time_column}`) AS observed_at,
  {_sql_literal(endpoint_name)} AS endpoint_name,
  COUNT(*) AS request_count,
  SUM(CASE WHEN status_code >= 400 THEN 1 ELSE 0 END) AS error_count,
  {latency_expression} AS latency_p50,
  {latency_p99_expression} AS latency_p99,
  {ttfb_expression} AS time_to_first_byte_p99
FROM {_sql_identifier(table_name)}
WHERE `{time_column}` >= timestampadd(HOUR, -{DASHBOARD_LOOKBACK_HOURS}, current_timestamp())
GROUP BY DATE_TRUNC('MINUTE', `{time_column}`)
""".strip()
        )
    return "\nUNION ALL\n".join(selects) if selects else None


metrics_query = f"""
SELECT
  scraped_at,
  endpoint_name,
  MAX(CASE WHEN chart_name = 'Response rate (per second)' THEN value END) AS response_rate,
  MAX(CASE WHEN chart_name = 'Response error rates (per second)' AND series_name = '4XX errors per second' THEN value END) AS error_4xx_rate,
  MAX(CASE WHEN chart_name = 'Response error rates (per second)' AND series_name = '5XX errors per second' THEN value END) AS error_5xx_rate,
  MAX(CASE WHEN chart_name = 'Latency (ms)' AND series_name = 'p50' THEN value END) AS latency_p50,
  MAX(CASE WHEN chart_name = 'Latency (ms)' AND series_name = 'p99' THEN value END) AS latency_p99,
  MAX(CASE WHEN chart_name = 'Model inference latency (ms)' AND series_name = 'p50' THEN value END) AS inference_p50,
  MAX(CASE WHEN chart_name = 'Model inference latency (ms)' AND series_name = 'p99' THEN value END) AS inference_p99,
  MAX(CASE WHEN chart_name = 'Request queue wait time (ms)' AND series_name = 'p50' THEN value END) AS queue_p50,
  MAX(CASE WHEN chart_name = 'Request queue wait time (ms)' AND series_name = 'p99' THEN value END) AS queue_p99,
  MAX(CASE WHEN chart_name = 'CPU Usage (%)' THEN value END) AS cpu_pct,
  MAX(CASE WHEN chart_name = 'Memory Usage (%)' THEN value END) AS mem_pct,
  MAX(CASE WHEN chart_name = 'Provisioned Concurrency' THEN value END) AS provisioned_concurrency
FROM model_serving_dashboard_series
WHERE chart_name IS NOT NULL
  AND scraped_at >= timestampadd(HOUR, -{DASHBOARD_LOOKBACK_HOURS}, current_timestamp())
GROUP BY scraped_at, endpoint_name
""".strip()

metrics_validation_query = metrics_query.replace(
    "FROM model_serving_dashboard_series",
    f"FROM {_sql_identifier(DASHBOARD_SERIES_VIEW)}",
)
metrics_validation_rows = spark.sql(metrics_validation_query).limit(1).collect()
if not metrics_validation_rows:
    raise RuntimeError(
        f"Dashboard metrics query returned no rows from {DASHBOARD_SERIES_VIEW} "
        f"for the last {DASHBOARD_LOOKBACK_HOURS} hours"
    )
_print_log(
    "dashboard_metrics_query_validated",
    returned_sample_row=True,
    source=DASHBOARD_SERIES_VIEW,
)
inference_query = _inference_query()
if inference_query:
    inference_validation_rows = spark.sql(inference_query).limit(1).collect()
    _print_log(
        "dashboard_inference_query_validated",
        returned_sample_row=bool(inference_validation_rows),
        source_tables=AVAILABLE_INFERENCE_TABLES,
    )
else:
    _print_log("dashboard_inference_query_skipped", reason="no_available_inference_tables")

## Build the dashboard definition

Assemble the overview page and add an inference-table page only when payload tables are configured and available.

In [ ]:
percent_format = {"type": "number", "decimalPlaces": {"type": "exact", "places": 1}}
rate_format = {"type": "number", "decimalPlaces": {"type": "max", "places": 4}}
ms_format = {"type": "number", "decimalPlaces": {"type": "max", "places": 1}}
count_format = {"type": "number", "abbreviation": "compact"}

datasets = [
    {
        "name": "ds_series",
        "displayName": "Serving metric series",
        "queryLines": _query_lines(metrics_query),
    }
]
overview_layout = [
    _text_widget("title", "## Model Serving metrics", 0, 0, 12, 1),
    _text_widget(
        "subtitle",
        f"Rolling {DASHBOARD_LOOKBACK_HOURS}-hour latency, request rates, errors, and replica usage.",
        0,
        1,
        12,
        1,
    ),
    _filter_widget(
        "ds_series", "endpoint_name", "Model endpoint", "filter_endpoint_name",
        "filter-single-select", 0, 2, 4,
    ),
    _filter_widget(
        "ds_series", "scraped_at", "Time range", "filter_scraped_at",
        "filter-date-range-picker", 4, 2, 8,
    ),
    _counter_widget(
        "ds_series", "scraped_at", "kpi_response_rate", "Response rate",
        "sum(response_rate)", "SUM(`response_rate`)", 0, 4, rate_format,
    ),
    _counter_widget(
        "ds_series", "scraped_at", "kpi_latency_p99", "Latency p99 (ms)",
        "avg(latency_p99)", "AVG(`latency_p99`)", 3, 4, ms_format,
    ),
    _counter_widget(
        "ds_series", "scraped_at", "kpi_error_5xx", "5XX errors / sec",
        "sum(error_5xx_rate)", "SUM(`error_5xx_rate`)", 6, 4, rate_format,
    ),
    _counter_widget(
        "ds_series", "scraped_at", "kpi_cpu", "CPU usage",
        "avg(cpu_pct)", "AVG(`cpu_pct`)", 9, 4, percent_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_latency", "Latency (ms)",
        [
            ("avg(latency_p50)", "AVG(`latency_p50`)", "p50"),
            ("avg(latency_p99)", "AVG(`latency_p99`)", "p99"),
        ],
        0, 8, ms_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_inference", "Model inference latency (ms)",
        [
            ("avg(inference_p50)", "AVG(`inference_p50`)", "p50"),
            ("avg(inference_p99)", "AVG(`inference_p99`)", "p99"),
        ],
        6, 8, ms_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_queue", "Request queue wait time (ms)",
        [
            ("avg(queue_p50)", "AVG(`queue_p50`)", "p50"),
            ("avg(queue_p99)", "AVG(`queue_p99`)", "p99"),
        ],
        0, 14, ms_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_response_rate", "Response rate (per second)",
        [("sum(response_rate)", "SUM(`response_rate`)", "response rate")],
        6, 14, rate_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_errors", "Response error rates (per second)",
        [
            ("sum(error_4xx_rate)", "SUM(`error_4xx_rate`)", "4XX errors per second"),
            ("sum(error_5xx_rate)", "SUM(`error_5xx_rate`)", "5XX errors per second"),
        ],
        0, 20, rate_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_cpu", "CPU Usage (%)",
        [("avg(cpu_pct)", "AVG(`cpu_pct`)", "average across replicas")],
        6, 20, percent_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_memory", "Memory Usage (%)",
        [("avg(mem_pct)", "AVG(`mem_pct`)", "average across replicas")],
        0, 26, percent_format,
    ),
    _line_widget(
        "ds_series", "scraped_at", "chart_concurrency", "Provisioned Concurrency",
        [
            (
                "avg(provisioned_concurrency)",
                "AVG(`provisioned_concurrency`)",
                "provisioned concurrency",
            )
        ],
        6, 26,
    ),
]
pages = [
    {
        "name": "overview",
        "displayName": "Overview",
        "pageType": "PAGE_TYPE_CANVAS",
        "layoutVersion": "GRID_V1",
        "layout": overview_layout,
    }
]

if inference_query:
    datasets.append(
        {
            "name": "ds_inference",
            "displayName": "Inference table request metrics",
            "queryLines": _query_lines(inference_query),
        }
    )
    pages.append(
        {
            "name": "inference",
            "displayName": "Inference tables",
            "pageType": "PAGE_TYPE_CANVAS",
            "layoutVersion": "GRID_V1",
            "layout": [
                _text_widget("inference_title", "## Inference table metrics", 0, 0, 12, 1),
                _text_widget(
                    "inference_subtitle",
                    "Request-level aggregates from inference tables discovered through the endpoint API.",
                    0, 1, 12, 1,
                ),
                _filter_widget(
                    "ds_inference", "endpoint_name", "Model endpoint", "inference_filter_endpoint",
                    "filter-single-select", 0, 2, 4,
                ),
                _filter_widget(
                    "ds_inference", "observed_at", "Time range", "inference_filter_time",
                    "filter-date-range-picker", 4, 2, 8,
                ),
                _counter_widget(
                    "ds_inference", "observed_at", "inference_requests", "Inference requests",
                    "sum(request_count)", "SUM(`request_count`)", 0, 4, count_format,
                ),
                _counter_widget(
                    "ds_inference", "observed_at", "inference_errors", "Inference errors",
                    "sum(error_count)", "SUM(`error_count`)", 3, 4, count_format,
                ),
                _counter_widget(
                    "ds_inference", "observed_at", "inference_latency_p99", "Inference latency p99 (ms)",
                    "avg(latency_p99)", "AVG(`latency_p99`)", 6, 4, ms_format,
                ),
                _counter_widget(
                    "ds_inference", "observed_at", "inference_ttfb_p99", "Time to first byte p99 (ms)",
                    "avg(time_to_first_byte_p99)", "AVG(`time_to_first_byte_p99`)", 9, 4, ms_format,
                ),
                _line_widget(
                    "ds_inference", "observed_at", "inference_latency", "Inference latency (ms)",
                    [
                        ("avg(latency_p50)", "AVG(`latency_p50`)", "p50"),
                        ("avg(latency_p99)", "AVG(`latency_p99`)", "p99"),
                    ],
                    0, 8, ms_format,
                ),
                _line_widget(
                    "ds_inference", "observed_at", "inference_volume", "Inference request volume",
                    [
                        ("sum(request_count)", "SUM(`request_count`)", "requests"),
                        ("sum(error_count)", "SUM(`error_count`)", "errors"),
                    ],
                    6, 8, count_format,
                ),
            ],
        }
    )

dashboard_definition = {
    "datasets": datasets,
    "pages": pages,
    "uiSettings": {
        "theme": {
            "canvasBackgroundColor": {"light": "#FCFCFC", "dark": "#1F272D"},
            "widgetBackgroundColor": {"light": "#FFFFFF", "dark": "#11171C"},
            "fontColor": {"light": "#11171C", "dark": "#E8ECF0"},
            "selectionColor": {"light": "#2272B4", "dark": "#8ACAFF"},
            "visualizationColors": [
                "#003F5C", "#594E90", "#BC4C96", "#FF5F66",
                "#FFA600", "#4E5185", "#99DDB4",
            ],
            "widgetHeaderAlignment": "LEFT",
        }
    },
}

## Publish the dashboard

Create or update the AI/BI dashboard, publish its latest revision, and report links to the configured resources.

In [ ]:
def _list_active_dashboards() -> list[dict[str, Any]]:
    dashboards: list[dict[str, Any]] = []
    page_token: str | None = None
    while True:
        query: dict[str, Any] = {"page_size": 1000, "show_trashed": False}
        if page_token:
            query["page_token"] = page_token
        response = workspace.api_client.do(
            "GET",
            "/api/2.0/lakeview/dashboards",
            query=query,
        )
        dashboards.extend(response.get("dashboards", []))
        page_token = response.get("next_page_token")
        if not page_token:
            return dashboards


if DASHBOARD_ID:
    existing_dashboard = workspace.api_client.do(
        "GET",
        f"/api/2.0/lakeview/dashboards/{DASHBOARD_ID}",
    )
    if existing_dashboard.get("lifecycle_state") == "TRASHED":
        raise RuntimeError(f"Configured DASHBOARD_ID is trashed: {DASHBOARD_ID}")
else:
    matching_dashboards = [
        dashboard
        for dashboard in _list_active_dashboards()
        if dashboard.get("display_name") == DASHBOARD_NAME
    ]
    if len(matching_dashboards) > 1:
        raise RuntimeError(
            f"Multiple active dashboards named {DASHBOARD_NAME!r} exist. "
            "Set DASHBOARD_ID in the first cell. Matches: "
            f"{[dashboard.get('dashboard_id') for dashboard in matching_dashboards]}"
        )
    existing_dashboard = matching_dashboards[0] if matching_dashboards else None
expected_stored_definition = json.loads(json.dumps(dashboard_definition))
for dataset in expected_stored_definition["datasets"]:
    dataset["catalog"] = STORAGE_CATALOG
    dataset["schema"] = STORAGE_SCHEMA

dashboard_body = {
    "display_name": DASHBOARD_NAME,
    "warehouse_id": resolved_warehouse_id,
    "parent_path": resolved_parent_path,
    "serialized_dashboard": json.dumps(dashboard_definition),
}
dashboard_query = {
    "dataset_catalog": STORAGE_CATALOG,
    "dataset_schema": STORAGE_SCHEMA,
}


def _get_dashboard(resolved_dashboard_id: str) -> dict[str, Any]:
    return workspace.api_client.do(
        "GET",
        f"/api/2.0/lakeview/dashboards/{resolved_dashboard_id}",
    )


def _dashboard_differences(dashboard: dict[str, Any]) -> list[dict[str, Any]]:
    try:
        stored_definition = json.loads(dashboard.get("serialized_dashboard") or "{}")
    except json.JSONDecodeError as error:
        return [{"path": "dashboard.serialized_dashboard", "error": str(error)}]
    expected_dashboard = {
        "display_name": DASHBOARD_NAME,
        "warehouse_id": resolved_warehouse_id,
        "parent_path": resolved_parent_path,
        "serialized_dashboard": expected_stored_definition,
    }
    actual_dashboard = {
        "display_name": dashboard.get("display_name"),
        "warehouse_id": dashboard.get("warehouse_id"),
        "parent_path": dashboard.get("parent_path"),
        "serialized_dashboard": stored_definition,
    }
    return _configuration_differences(
        actual_dashboard,
        expected_dashboard,
        path="dashboard",
    )


def _create_dashboard() -> dict[str, Any]:
    response = workspace.api_client.do(
        "POST",
        "/api/2.0/lakeview/dashboards",
        query=dashboard_query,
        body=dashboard_body,
    )
    if not response.get("dashboard_id"):
        raise RuntimeError(f"Dashboard create response did not contain dashboard_id: {response}")
    return response


if existing_dashboard:
    dashboard_id = existing_dashboard["dashboard_id"]
    stored_dashboard = _get_dashboard(dashboard_id)
    dashboard_differences = _dashboard_differences(stored_dashboard)
else:
    dashboard_id = None
    stored_dashboard = None
    dashboard_differences = []
_print_log(
    "dashboard_configuration_resolved",
    dashboard_name=DASHBOARD_NAME,
    existing_dashboard_id=dashboard_id,
    differences=dashboard_differences,
    warehouse_id=resolved_warehouse_id,
    parent_path=resolved_parent_path,
    dataset_names=[dataset["name"] for dataset in dashboard_definition["datasets"]],
    page_names=[page["name"] for page in dashboard_definition["pages"]],
)
if dashboard_id is None:
    created_dashboard = _create_dashboard()
    dashboard_id = created_dashboard["dashboard_id"]
    dashboard_action = "created"
elif not dashboard_differences:
    dashboard_action = "unchanged"
else:
    try:
        workspace.api_client.do(
            "PATCH",
            f"/api/2.0/lakeview/dashboards/{dashboard_id}",
            query=dashboard_query,
            body=dashboard_body,
        )
        stored_dashboard = _get_dashboard(dashboard_id)
        remaining_dashboard_differences = _dashboard_differences(stored_dashboard)
    except BadRequest as error:
        remaining_dashboard_differences = [
            {"path": "dashboard.update", "error": str(error)}
        ]
    if remaining_dashboard_differences:
        old_dashboard_id = dashboard_id
        _print_log(
            "dashboard_rebuild_required",
            level="warning",
            old_dashboard_id=old_dashboard_id,
            differences=remaining_dashboard_differences,
        )
        workspace.api_client.do(
            "DELETE",
            f"/api/2.0/lakeview/dashboards/{old_dashboard_id}",
        )
        created_dashboard = _create_dashboard()
        dashboard_id = created_dashboard["dashboard_id"]
        dashboard_action = "rebuilt"
    else:
        dashboard_action = "updated"

stored_dashboard = _get_dashboard(dashboard_id)
final_dashboard_differences = _dashboard_differences(stored_dashboard)
if final_dashboard_differences:
    raise RuntimeError(
        "Dashboard settings did not converge after reconciliation: "
        + json.dumps(final_dashboard_differences, sort_keys=True)
    )
_print_log(
    "dashboard_draft_applied",
    action=dashboard_action,
    dashboard_id=dashboard_id,
    catalog=STORAGE_CATALOG,
    schema=STORAGE_SCHEMA,
)

published_dashboard: dict[str, Any] | None = None
if dashboard_action == "unchanged":
    try:
        published_dashboard = workspace.api_client.do(
            "GET",
            f"/api/2.0/lakeview/dashboards/{dashboard_id}/published",
        )
    except NotFound:
        published_dashboard = None
if published_dashboard is None:
    published_dashboard = workspace.api_client.do(
        "POST",
        f"/api/2.0/lakeview/dashboards/{dashboard_id}/published",
        body={
            "embed_credentials": True,
            "warehouse_id": resolved_warehouse_id,
        },
    )
    publish_action = "published"
else:
    publish_action = "unchanged"
if not published_dashboard.get("revision_create_time"):
    raise RuntimeError(
        f"Dashboard publish response did not contain revision_create_time: {published_dashboard}"
    )
_print_log(
    "dashboard_publication_verified",
    action=publish_action,
    dashboard_id=dashboard_id,
    revision_create_time=published_dashboard["revision_create_time"],
    warehouse_id=resolved_warehouse_id,
)

_print_log(
    "setup_complete",
    include_endpoint_names=INCLUDE_ENDPOINT_NAMES,
    discovered_non_foundation_endpoints=RESOLVED_ENDPOINT_NAMES,
    configured_inference_tables=CONFIGURED_INFERENCE_TABLES,
    available_inference_tables=AVAILABLE_INFERENCE_TABLES,
    inference_table_status=INFERENCE_TABLE_STATUS,
    job={
        "action": job_action,
        "job_id": job_id,
        "mode": "continuous",
        "performance_target": COLLECTOR_PERFORMANCE_TARGET,
        "poll_interval_seconds": COLLECTION_INTERVAL_SECONDS,
        "url": f"{WORKSPACE_HOST}/jobs/{job_id}",
    },
    dashboard={
        "action": dashboard_action,
        "publish_action": publish_action,
        "dashboard_id": dashboard_id,
        "published_at": published_dashboard.get("revision_create_time"),
        "url": f"{WORKSPACE_HOST}/dashboardsv3/{dashboard_id}/published",
    },
)